# Inera Edit Flow — Real-face lip-sync (free Google Colab GPU)

Turn **your own face** (a short video or one photo) + **your own voice** (or the voice made in Inera's Avatar Studio) into a talking video with accurate lip-sync.

* Model: **LatentSync 1.5** by ByteDance — code and weights Apache-2.0. Code: https://github.com/bytedance/LatentSync
* **Licence warning (read before commercial use):** LatentSync finds the face with **InsightFace "buffalo_l"** models, which it downloads automatically. InsightFace states those pretrained models are for **non-commercial research only**. Personal / non-commercial use is fine; for paid client work, check InsightFace's licence or ask them for a commercial licence first.
* Runs on the **free T4 GPU** in Google Colab. No API key, no payment. You only need a Google account to use Colab itself.
* Your files go to *your own* Colab session (Google's computer), not to Inera.

**Honest status:** this notebook follows LatentSync's official install and inference scripts, but Inera's builder could not run it on a GPU while building it. If a step fails, the error appears under that cell — copy it and ask for help.

### How to use
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the cells **in order** (click ▶ on each, or Runtime → Run all).
3. Step 3 asks you to upload: **face** (MP4/MOV/WebM video of you talking or sitting still, 5–30 s, face clearly visible, front-facing — or a JPG/PNG photo) and **voice** (WAV/MP3/M4A/WebM, or a video whose sound will be used).
4. The result downloads automatically at the end.

**Tips for best quality:** good light on the face, face larger than ~1/4 of the frame, no hands over the mouth, a neutral/closed mouth in the source clip. A video source looks far more natural than a single photo (a photo only moves the lips).

### Rules — please read
Only use a face and voice **you own or have written permission to use**. Do not create content that deceives people, impersonates someone, or is used for fraud or harassment. The output is labelled **AI-generated** on screen and in the file's metadata.

## Step 1 — Consent (required)

In [ ]:
#@title Confirm consent, then run this cell
I_AM_THE_PERSON_OR_HAVE_WRITTEN_PERMISSION = False #@param {type:"boolean"}
I_WILL_NOT_USE_IT_TO_DECEIVE_OR_IMPERSONATE = False #@param {type:"boolean"}
if not (I_AM_THE_PERSON_OR_HAVE_WRITTEN_PERMISSION and I_WILL_NOT_USE_IT_TO_DECEIVE_OR_IMPERSONATE):
    raise SystemExit("Tick both boxes above (they confirm you have the right to use this face and voice), then run this cell again.")
CONSENT_OK = True
print("Thanks. Consent recorded for this session.")

## Step 2 — Install LatentSync (about 5–8 minutes, once per session)

In [ ]:
#@title Install (GPU check, code, Python 3.10 environment, model weights)
import subprocess, sys, os, shutil
assert globals().get("CONSENT_OK"), "Run Step 1 first."
def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = "\n".join(r.stdout.strip().splitlines()[-25:])
    if r.returncode != 0:
        print(tail); raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")
    return r.stdout

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, text=True, capture_output=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU found. Menu: Runtime -> Change runtime type -> T4 GPU -> Save, then run again.")
print("GPU:", gpu)

os.chdir("/content")
if not os.path.isdir("/content/LatentSync"):
    sh("git clone --depth 1 https://github.com/bytedance/LatentSync.git")
os.chdir("/content/LatentSync")

# LatentSync pins Python 3.10 packages; Colab's system Python is newer, so build an isolated 3.10 env with uv.
sh(f"{sys.executable} -m pip install -q uv")
if not os.path.exists("/content/ls-env/bin/python"):
    sh("uv venv -q -p 3.10 /content/ls-env")
sh("VIRTUAL_ENV=/content/ls-env uv pip install -q -r requirements.txt")
sh("VIRTUAL_ENV=/content/ls-env uv pip install -q 'huggingface-hub==0.30.2'")

# LatentSync 1.5 weights (needs ~8 GB GPU memory; version 1.6 needs 18 GB and does not fit a free T4)
os.makedirs("checkpoints/whisper", exist_ok=True)
dl = "/content/ls-env/bin/python -c \"from huggingface_hub import hf_hub_download as d; import sys; d('ByteDance/LatentSync-1.5', sys.argv[1], local_dir='checkpoints')\""
for f in ["whisper/tiny.pt", "latentsync_unet.pt"]:
    if not os.path.exists(os.path.join("checkpoints", f)):
        sh(f"{dl} {f}")
for f in ["checkpoints/whisper/tiny.pt", "checkpoints/latentsync_unet.pt", "configs/unet/stage2.yaml"]:
    assert os.path.exists(f), f"Missing {f} — the LatentSync repository layout may have changed."
print("\nInstalled. Continue with Step 3.")

## Step 3 — Upload your face and your voice

In [ ]:
#@title Upload face (video or photo), then voice (audio or video)
from google.colab import files
import os, shutil
os.makedirs("/content/in", exist_ok=True)
print("1) Choose your FACE file (video or photo):")
up = files.upload()
assert up, "No face file uploaded."
FACE = "/content/in/face_" + list(up.keys())[0].replace(" ", "_")
shutil.move(list(up.keys())[0], FACE)
print("2) Choose your VOICE file (audio, or a video whose sound to use):")
up = files.upload()
assert up, "No voice file uploaded."
VOICE = "/content/in/voice_" + list(up.keys())[0].replace(" ", "_")
shutil.move(list(up.keys())[0], VOICE)
print("Face:", FACE); print("Voice:", VOICE)

## Step 4 — Prepare inputs (match lengths, 25 fps, 16 kHz voice)

In [ ]:
#@title Prepare
import subprocess, json, os
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0: print(r.stderr[-2000:]); raise RuntimeError("ffmpeg/ffprobe failed: " + cmd)
    return r.stdout
def duration(p):
    return float(json.loads(sh(f'ffprobe -v error -show_entries format=duration -of json "{p}"'))["format"]["duration"])
os.makedirs("/content/work", exist_ok=True)
AUDIO = "/content/work/voice16k.wav"
sh(f'ffmpeg -y -v error -i "{VOICE}" -vn -ac 1 -ar 16000 "{AUDIO}"')
AD = duration(AUDIO)
print(f"Voice length: {AD:.1f} s")
if AD > 600: print("Note: long audio takes a long time on a free T4 (roughly 1–3 minutes per 10 seconds). Colab may disconnect after a few hours.")
is_photo = FACE.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
VIDEO = "/content/work/face25.mp4"
if is_photo:
    # still photo -> video of the same length (very slow, subtle zoom so it does not look frozen)
    iw, ih = map(int, sh(f'ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of csv=p=0 "{FACE}"').strip().split(",")[:2])
    H = min(720, ih - ih % 2); W = int(round(H * iw / ih / 2)) * 2
    sh(f'ffmpeg -y -v error -loop 1 -i "{FACE}" -t {AD + 0.5:.2f} -vf "scale={W}:{H},zoompan=z=\'min(1.0+0.0004*on,1.06)\':x=\'iw/2-iw/zoom/2\':y=\'ih/2-ih/zoom/2\':d=1:s={W}x{H}:fps=25" -pix_fmt yuv420p -an "{VIDEO}"')
else:
    base = "/content/work/face_base.mp4"
    sh(f'ffmpeg -y -v error -i "{FACE}" -vf "fps=25,scale=-2:\'min(720,ih)\'" -an -pix_fmt yuv420p "{base}"')
    VD = duration(base)
    if VD + 0.05 < AD:
        # forward + backward loop (no visible jump), repeated to cover the whole voice
        sh(f'ffmpeg -y -v error -i "{base}" -vf reverse -an "/content/work/face_rev.mp4"')
        with open("/content/work/list.txt", "w") as f:
            for _ in range(int(AD // (2 * VD)) + 1):
                f.write("file '/content/work/face_base.mp4'\nfile '/content/work/face_rev.mp4'\n")
        sh(f'ffmpeg -y -v error -f concat -safe 0 -i /content/work/list.txt -t {AD + 0.2:.2f} -c:v libx264 -pix_fmt yuv420p "{VIDEO}"')
    else:
        sh(f'ffmpeg -y -v error -i "{base}" -t {AD + 0.2:.2f} -c:v libx264 -pix_fmt yuv420p "{VIDEO}"')
print("Face video:", VIDEO, f"{duration(VIDEO):.1f} s")

## Step 5 — Generate the lip-synced video

In [ ]:
#@title Run LatentSync 1.5
import subprocess, os, time
os.chdir("/content/LatentSync")
QUALITY_STEPS = 20 #@param {type:"slider", min:10, max:50, step:5}
GUIDANCE = 1.5 #@param {type:"slider", min:1.0, max:3.0, step:0.1}
RAW = "/content/work/lipsync_raw.mp4"
t0 = time.time()
cmd = ["/content/ls-env/bin/python", "-m", "scripts.inference",
       "--unet_config_path", "configs/unet/stage2.yaml",
       "--inference_ckpt_path", "checkpoints/latentsync_unet.pt",
       "--inference_steps", str(QUALITY_STEPS), "--guidance_scale", str(GUIDANCE), "--enable_deepcache",
       "--video_path", VIDEO, "--audio_path", AUDIO, "--video_out_path", RAW]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    line = line.rstrip()
    if line: print(line[-200:])
p.wait()
if p.returncode != 0 or not os.path.exists(RAW):
    raise RuntimeError("LatentSync failed. Most common causes: no face found in some frames (use a clip where the face is always visible), or the GPU ran out of memory (use a shorter clip / lower resolution).")
print(f"Done in {(time.time()-t0)/60:.1f} min")

## Step 6 — Label as AI-generated and download

In [ ]:
#@title Finish
import subprocess, os
from google.colab import files
OUT = "/content/work/inera-realface-lipsync.mp4"
label = "drawtext=text='AI-generated':x=w-tw-20:y=h-th-20:fontsize=h/40:fontcolor=white@0.75:box=1:boxcolor=black@0.35:boxborderw=8"
font = "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"
if os.path.exists(font): label = label.replace("drawtext=", f"drawtext=fontfile={font}:")
r = subprocess.run(f'ffmpeg -y -v error -i "{RAW}" -i "{VOICE}" -map 0:v -map 1:a? -vf "{label}" -c:v libx264 -crf 18 -preset medium -c:a aac -b:a 192k -shortest -metadata comment="AI-generated lip-sync (LatentSync 1.5) made with Inera Edit Flow" "{OUT}"', shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print("Could not draw the on-screen label (", r.stderr[-300:], ") — keeping the metadata label only.")
    subprocess.run(f'ffmpeg -y -v error -i "{RAW}" -c copy -metadata comment="AI-generated lip-sync (LatentSync 1.5) made with Inera Edit Flow" "{OUT}"', shell=True, check=True)
print("Saved", OUT)
files.download(OUT)

### Next
* Bring the downloaded video back into **Inera Edit Flow → Video Editor** to add captions, topic visuals, music and a hook automatically.
* Run Step 3 → 6 again for another video (no reinstall needed while the session is alive).

Credits: LatentSync © ByteDance (Apache-2.0) · Whisper tiny audio encoder © OpenAI (MIT) · VAE `stabilityai/sd-vae-ft-mse` © Stability AI (MIT) · face detection: InsightFace buffalo_l (non-commercial research licence — see the warning at the top).